# EDA — Movilidad Medellín

Análisis exploratorio de datos de movilidad de Medellín.  
Todo el análisis usa funciones de `src/pipeline.py`; no se duplica lógica.

**Datasets integrados:**
- `velocidad_e_intensidad_vehicular_en_medellin.csv` — Fuente primaria del ICV
- `Aforos_Vehiculares.csv` — Conteos vehiculares con coordenadas WGS84
- `simmtrafficdata.csv` — Sensores CCTV en corredores
- `Pasajeros_movilizados.csv` — Pasajeros Metro 2014-2021
- `Ciclorrutas.csv` — Red ciclista existente y proyectada
- `proyecciones_de_poblacion_medellin_2019.csv` — Población por comuna

In [ ]:
import sys
from pathlib import Path

# Añadir raíz del proyecto al path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from src.pipeline import (
    load_config, load_all_data, run_etl, build_criticality_index,
    eda_descriptive_by_corridor, eda_descriptive_by_commune,
    eda_temporal_heatmap, eda_speed_vs_intensity, eda_peak_hour_bar,
    eda_congestion_zones, eda_top_problematic_corridors,
    eda_passenger_trend, eda_ciclorrutas_bar,
)

print('Imports OK')

## 1. Ingesta y ETL

In [ ]:
config = load_config(str(ROOT / 'config.yaml'))
config['data']['raw_dir'] = str(ROOT / 'data' / 'raw')

raw = load_all_data(config)
print('Datasets cargados:', {k: v.shape for k, v in raw.items() if not v.empty})

In [ ]:
cleaned = run_etl(raw, config)

df_vel  = cleaned['velocidad']
df_simm = cleaned['simm']
df_afo  = cleaned['aforos']
df_pax  = cleaned['pasajeros']
df_cic  = cleaned['ciclorrutas']
df_pop  = cleaned['poblacion']

print('ETL completado.')
for k, v in cleaned.items():
    print(f'  {k}: {v.shape}')

## 2. Vista general de los datasets

In [ ]:
print('=== velocidad_e_intensidad ===')
df_vel.head(3)

In [ ]:
print('Columnas velocidad:', df_vel.columns.tolist())
df_vel[['corredor','hora','velocidad_km_h','intensidad','nombre_comuna']].describe()

In [ ]:
print('=== simmtrafficdata ===')
df_simm[['corredor','fechahora','velocidad','intensidad','lon','lat']].head(3)

## 3. EDA — Análisis descriptivo por corredor

In [ ]:
corr_stats = eda_descriptive_by_corridor(df_vel)
print(f'Total corredores: {len(corr_stats)}')
corr_stats.head(10)

In [ ]:
# Top 10 corredores más LENTOS (mayor congestión)
corr_stats.sort_values('vel_media').head(10)[['corredor','vel_media','int_media','n_obs']]

## 4. EDA — Análisis por comuna

In [ ]:
comm_stats = eda_descriptive_by_commune(df_vel)
comm_stats.head(10)

## 5. EDA — Intensidad por hora del día

In [ ]:
fig = eda_peak_hour_bar(df_vel, config)
plt.show()

**Observación:** Las barras rojas indican horas pico (6-9h AM y 17-20h PM) donde la intensidad vehicular es más alta. El tercer pico del mediodía (12-13h) también es notable.

## 6. EDA — Heatmap intensidad × hora × día

In [ ]:
fig = eda_temporal_heatmap(df_vel)
plt.show()

**Observación:** El heatmap revela los días y horas con mayor congestión. Los lunes y viernes tienden a tener picos más pronunciados en la tarde.

## 7. EDA — Relación velocidad vs intensidad

In [ ]:
fig = eda_speed_vs_intensity(df_vel)
plt.show()

**Observación:** Se confirma la relación inversa esperada: a mayor intensidad (flujo vehicular), menor velocidad. La franja de mañana (rojo) concentra los puntos de mayor congestión.

## 8. EDA — Detección de zonas con baja velocidad y alto flujo

In [ ]:
cong_zones = eda_congestion_zones(df_vel, config)
print(f'Corredores con congestión simultánea: {len(cong_zones)}')
cong_zones.head(10)

In [ ]:
top_prob = eda_top_problematic_corridors(df_vel, n=10)
print('Top 10 corredores más problemáticos (ratio Intensidad/Velocidad):')
top_prob[['corredor','vel_media','int_media','ratio_iv']]

## 9. EDA — Pasajeros movilizados (Metro Medellín)

In [ ]:
fig = eda_passenger_trend(df_pax)
plt.show()

**Observación:** Se observa la caída drástica de pasajeros en 2020 por COVID-19 y la recuperación gradual en 2021. El crecimiento sostenido 2014-2019 indica alta dependencia del sistema metro.

## 10. EDA — Ciclorrutas

In [ ]:
fig = eda_ciclorrutas_bar(df_cic)
plt.show()
print(df_cic['estado'].value_counts())

## 11. Índice de Criticidad Vial (ICV)

In [ ]:
icv_results = build_criticality_index(
    df_vel, df_simm, df_afo, df_pop, config
)

top10    = icv_results['top10_corridors']
full_icv = icv_results['full_index']
comm_pressure = icv_results['commune_pressure']

print('=== TOP 10 CORREDORES MÁS CRÍTICOS ENTRE SEMANA ===')
top10

In [ ]:
print('=== COMUNAS CON MAYOR PRESIÓN VEHICULAR EN HORA PICO ===')
comm_pressure.head(10)

In [ ]:
# Visualización del ICV
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

if not full_icv.empty:
    top20 = full_icv.head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(top20['corredor'], top20['icv'],
                   color=['#e74c3c' if v >= 70 else '#f39c12' if v >= 40 else '#27ae60'
                          for v in top20['icv']])
    ax.set_xlabel('Índice de Criticidad Vial (0-100)')
    ax.set_title('Top 20 Corredores por ICV (entre semana, hora pico)', fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    for bar, val in zip(bars, top20['icv']):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

## 12. Conclusiones

### Hallazgos principales:

1. **Corredores críticos**: Los 10 corredores con mayor ICV concentran las combinaciones más adversas de baja velocidad + alta intensidad en hora pico entre semana.

2. **Franjas horarias**: Los picos AM (7-8h) y PM (17-18h) son consistentes en todos los corredores. El mediodía muestra un tercer pico significativo.

3. **Relación velocidad-intensidad**: La correlación negativa confirma que los cuellos de botella son estructurales, no aleatorios.

4. **Transporte público**: El metro muestra crecimiento sostenido hasta 2019 y recuperación parcial post-COVID. Ampliar cobertura puede aliviar la demanda vehicular.

5. **Ciclorrutas**: La mayoría de tramos están proyectados, no construidos. Ejecutarlos en zonas de alta congestión tiene alto potencial de impacto.

### Recomendaciones de datos:
- El dataset de aforos (2018) tiene desfase temporal con velocidad (2020+); actualizar con datos más recientes mejoraría la precisión del ICV.
- Las coordenadas proyectadas en velocidad_e_intensidad (MAGNA-SIRGAS) requieren transformación para visualización cartográfica.